In [ ]:
import csv
import gc
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

os.environ["MALLOC_ARENA_MAX"] = "2"

EPOCHS = 6
BATCH = 4
LR = 2e-4
LR_REANUDAR = 5e-5
POS_WEIGHT = 5.0
WORKERS = min(4, os.cpu_count() or 2)
WARMUP = 300
MAX_HOURS = 11.0
UMBRAL = 0.3
REANUDAR = "/kaggle/input/models/manuelht311/last-keypoints-2/tensorflow2/default/1/lastKeypoints2.pt"
BN_LIBRE = False
SMOKE = False
OUT = Path("/kaggle/working/out")

if SMOKE:
    EPOCHS = 1

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yacs"])
HRNET = Path("/kaggle/working/HRNet")
if not HRNET.exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/HRNet/HRNet-Human-Pose-Estimation.git", str(HRNET)])
sys.path.insert(0, str(HRNET / "lib"))

import cv2
cv2.setNumThreads(0)
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from config import cfg
from models import pose_hrnet

assert torch.cuda.is_available(), "Activa la GPU en Settings > Accelerator"
dev = torch.device("cuda")
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU)

DATA = None
for mp in Path("/kaggle/input").rglob("metadata.json"):
    try:
        meta = json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        continue
    if meta.get("num_classes") == 28 and (mp.parent / "line_maps" / "train").is_dir():
        DATA = mp.parent
        break
assert DATA is not None, "Adjunta el dataset soccernet-calibration-2023-hrnet-lines"
NC = meta["num_classes"]
IW, IH = meta["input_size"]
OW, OH = meta["output_size"]
print("Dataset:", DATA, "| entrada", (IW, IH), "| salida", (OW, OH))

MEAN_T = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1) * 255.0
STD_T = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1) * 255.0


def flag_lectura(ruta):
    im = cv2.imread(ruta)
    h0, w0 = im.shape[:2]
    for f, flag in ((8, cv2.IMREAD_REDUCED_COLOR_8),
                    (4, cv2.IMREAD_REDUCED_COLOR_4),
                    (2, cv2.IMREAD_REDUCED_COLOR_2)):
        if w0 // f >= IW and h0 // f >= IH:
            return flag
    return cv2.IMREAD_COLOR


class LineMaps(Dataset):
    def __init__(self, split):
        imgs = {p.stem: str(p) for p in (DATA / "images" / split).iterdir() if p.is_file()}
        maps = [m for m in sorted((DATA / "line_maps" / split).glob("*.npz")) if m.stem in imgs]
        if SMOKE:
            maps = maps[:64]
        assert maps, f"Sin muestras en {split}"
        self.maps = np.array([str(m) for m in maps], dtype=np.bytes_)
        self.imgs = np.array([imgs[m.stem] for m in maps], dtype=np.bytes_)
        self.flag = flag_lectura(self.imgs[0].decode())

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, i):
        img = cv2.imread(self.imgs[i].decode(), self.flag)
        if img.shape[1] != IW or img.shape[0] != IH:
            img = cv2.resize(img, (IW, IH), interpolation=cv2.INTER_AREA)
        with np.load(self.maps[i].decode()) as z:
            tgt = np.ascontiguousarray(z["line_maps"].astype(np.uint8, copy=False))
        return torch.from_numpy(img), torch.from_numpy(tgt)


def cargador(split, train, workers):
    return DataLoader(LineMaps(split), batch_size=BATCH, shuffle=train, drop_last=train,
                      num_workers=workers, pin_memory=True, persistent_workers=True,
                      prefetch_factor=4)


train_dl = cargador("train", True, WORKERS)
val_dl = cargador("val", False, max(2, WORKERS // 2))
print(f"train={len(train_dl.dataset):,} val={len(val_dl.dataset):,} | {len(train_dl)} pasos/epoch")
print(f"Lectura JPEG con flag {train_dl.dataset.flag} | workers {WORKERS}")


def preparar(x, y, train):
    x = x.to(dev, non_blocking=True).float().flip(3).permute(0, 3, 1, 2)
    y = y.to(dev, non_blocking=True).float().div_(255.0)
    if train:
        b = x.shape[0]
        m = (torch.rand(b, 1, 1, 1, device=dev) < 0.35).float()
        a = 1.0 + m * (torch.rand(b, 1, 1, 1, device=dev) * 0.30 - 0.15)
        s = m * (torch.rand(b, 1, 1, 1, device=dev) * 24.0 - 12.0)
        x = (x * a + s).clamp_(0, 255)
    x = (x - MEAN_T) / STD_T
    return x, y


cfg.defrost()
cfg.merge_from_file(str(HRNET / "experiments/coco/hrnet/w32_256x192_adam_lr1e-3.yaml"))
cfg.MODEL.NUM_JOINTS = NC
cfg.MODEL.IMAGE_SIZE = [IW, IH]
cfg.MODEL.HEATMAP_SIZE = [OW, OH]
cfg.MODEL.INIT_WEIGHTS = True
cfg.MODEL.PRETRAINED = ""
cfg.freeze()
net = pose_hrnet.get_pose_net(cfg, is_train=True)

ckpt = next(iter(Path("/kaggle/input").rglob("pose_hrnet_w32_256x192.pth")), None)
if ckpt:
    sd = torch.load(ckpt, map_location="cpu", weights_only=False)
    sd = {k.removeprefix("module."): v for k, v in sd.items()}
    ms = net.state_dict()
    ok = {k: v for k, v in sd.items() if k in ms and ms[k].shape == v.shape}
    net.load_state_dict(ok, strict=False)
    print(f"Preentrenado cargado: {len(ok)} tensores")
else:
    print("Sin pesos preentrenados: entrenando desde cero")

BEST0 = -1.0
START_EP = 0
st = None
RES = None
if not SMOKE and REANUDAR:
    p = Path(REANUDAR)
    if p.is_file():
        RES = p
    else:
        RES = next(iter(Path("/kaggle/input").rglob(p.name)), None)
    if RES is None:
        print(f"No encuentro {REANUDAR}: entreno sin reanudar")
if RES:
    st = torch.load(RES, map_location="cpu", weights_only=False)
    net.load_state_dict(st["state_dict"])
    BEST0 = float(st.get("best_val_macro_dice", -1.0))
    START_EP = int(st.get("epoch", 0))
    print(f"REANUDO desde {RES} (epochs hechos: {START_EP}, mejor Dice: {BEST0:.4f})")

CONGELAR_BN = (ckpt is not None or RES is not None) and not BN_LIBRE


def congelar_bn(m):
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.eval()


print("BatchNorm congelada" if CONGELAR_BN else "BatchNorm libre")


class AMP(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    PAD_H = (-IH) % 32
    PAD_W = (-IW) % 32

    def forward(self, x):
        if self.PAD_H or self.PAD_W:
            x = F.pad(x, (0, self.PAD_W, 0, self.PAD_H))
        x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16):
            out = self.m(x)
        return out[:, :, :OH, :OW]


net = net.to(memory_format=torch.channels_last)
model = AMP(net).to(dev)
if NGPU > 1:
    model = nn.DataParallel(model)

LR_BASE = LR_REANUDAR if RES else LR
opt = torch.optim.AdamW(net.parameters(), lr=LR_BASE, weight_decay=1e-4)
if st is not None and "opt" in st:
    try:
        opt.load_state_dict(st["opt"])
        print("Estado del optimizador restaurado")
    except Exception as e:
        print("No se pudo restaurar el optimizador:", e)
elif RES:
    print(f"Checkpoint sin optimizador: LR de reanudacion {LR_BASE:g}")
scaler = torch.amp.GradScaler("cuda")
torch.backends.cudnn.benchmark = True

PASOS = len(train_dl)
TOTAL = EPOCHS * PASOS


def factor_lr(s):
    if s < WARMUP:
        return 0.1 + 0.9 * s / WARMUP
    p = min(1.0, (s - WARMUP) / max(1, TOTAL - WARMUP))
    return 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * p))


def loss_fn(logits, tgt):
    logits = logits.float()
    bce = F.binary_cross_entropy_with_logits(
        logits, tgt, pos_weight=torch.tensor(POS_WEIGHT, device=logits.device))
    p = torch.sigmoid(logits)
    inter = (p * tgt).sum((2, 3))
    den = p.sum((2, 3)) + tgt.sum((2, 3))
    d = (2 * inter + 1) / (den + 1)
    act = (tgt.sum((2, 3)) > 0).float()
    dice = ((1 - d) * act).sum() / act.sum().clamp(min=1)
    return bce + 0.5 * dice


def make_pack(ep, bd, con_opt):
    pack = {"state_dict": net.state_dict(), "epoch": ep, "best_val_macro_dice": bd,
            "num_classes": NC, "class_names": meta["classes"], "input_size": [IW, IH],
            "output_size": [OW, OH]}
    if con_opt:
        pack["opt"] = opt.state_dict()
    return pack


OUT.mkdir(parents=True, exist_ok=True)
(OUT / "metrics.csv").write_text("epoch,train_loss,val_loss,val_macro_dice,minutes\n")
best, t_global = BEST0, time.time()
paso = 0
FIN = START_EP + EPOCHS

for ep in range(START_EP + 1, FIN + 1):
    t0 = time.time()
    model.train()
    if CONGELAR_BN:
        congelar_bn(net)
    tl = torch.zeros((), device=dev)
    n = 0
    t_last, n_last = t0, 0
    for x, y in train_dl:
        x, y = preparar(x, y, True)
        for g in opt.param_groups:
            g["lr"] = LR_BASE * factor_lr(paso)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()
        tl += loss.detach()
        n += 1
        paso += 1
        if n % 200 == 0 or n == 10:
            now = time.time()
            rate = (n - n_last) / (now - t_last)
            t_last, n_last = now, n
            print(f"  ep {ep} paso {n}/{PASOS} | loss {tl.item() / n:.4f} | {rate:.2f} it/s | "
                  f"lr {opt.param_groups[0]['lr']:.2e} | quedan ~{(PASOS - n) / rate / 60:.0f} min | "
                  f"RAM {psutil.virtual_memory().percent:.0f}%", flush=True)
        if n % 1000 == 0:
            torch.save(make_pack(ep - 1, best, True), OUT / "last.pt")
    train_loss = tl.item() / max(n, 1)

    model.eval()
    vl, vn = 0.0, 0
    inter = torch.zeros(NC, device=dev)
    den = torch.zeros(NC, device=dev)
    pos = torch.zeros(NC, device=dev)
    with torch.no_grad():
        for x, y in val_dl:
            x, y = preparar(x, y, False)
            lg = model(x).float()
            vl += loss_fn(lg, y).item()
            vn += 1
            pr, gt = torch.sigmoid(lg) >= UMBRAL, y >= 0.5
            inter += (pr & gt).sum((0, 2, 3)).float()
            den += pr.sum((0, 2, 3)).float() + gt.sum((0, 2, 3)).float()
            pos += gt.sum((0, 2, 3)).float()
    act = pos > 0
    dice = (((2 * inter + 1) / (den + 1))[act]).mean().item()

    mins = (time.time() - t0) / 60
    print(f"Epoch {ep}/{FIN} | train {train_loss:.4f} | val {vl / vn:.4f} | Dice {dice:.4f} | {mins:.1f} min", flush=True)
    with (OUT / "metrics.csv").open("a") as f:
        csv.writer(f).writerow([ep, train_loss, vl / vn, dice, mins])

    torch.save(make_pack(ep, max(best, dice), True), OUT / "last.pt")
    if dice > best:
        best = dice
        torch.save(make_pack(ep, best, False), OUT / "best.pt")

    gc.collect()
    torch.cuda.empty_cache()
    if (time.time() - t_global) / 3600 > MAX_HOURS:
        print("Limite de tiempo alcanzado, paro aqui.")
        break

print(f"Hecho. Mejor Dice macro: {best:.4f}. Pesos en {OUT}/best.pt")